In [2]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'GEMINI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [3]:

from google import genai
from typing import List,Dict

client=genai.Client()
def generate_response(prompt: List[Dict]) -> str:
    texts="\n".join(
        messages['content']
        for messages in prompt)
    res = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=texts,
    )

    return res.text



In [4]:
import json
agent_rules=[{
    "role":"system",
    "content":"""
    you are an ai agent that can perform tasks using avaliable tools
    avaliable tools:
    -list_files() ->List[str]: list all files in the current directory
    -read_file(file_name:str)->str: read the content of a file
    - terminate(message :str)end the agent loop and print a small summary to the user.

    if a user asks about files, list them before reading.

    Every action must be an action.
    Respond in this formate

    ```action
    {
        "tool_name":"insert tool_name",
        "args":{...fill in any arguments here...}
    }
    """,
}]

memory = [
    {"role": "user", "content": "What files are in this directory?"},
    {"role":"system","content":"academic/ text1.txt text2.txt text3.txt, code/ a.cpp b.cpp c.cpp"},

]

def extract_markdown_block(response,par):
  if "```" not in response:
    return response.strip()
  block=response.split("```")[1].strip()
  return block[(len("action")):].strip()

def parse_action(response: str)->Dict:
  try:
    response=extract_markdown_block(response,"action")
    response_json=json.loads(response)
    if "tool_name" in response_json and "args" in response_json:
      return response_json
    else :
      return {"tool_name":"error","args":{"message":"you must respond with a json tool invocation."}}

  except json.JSONDecodeError:
    return {"tool_name":"error","args":{"message":"Invalid json response,you must respond with a json too invocatiion."}}



def list_file():
  files=["text1.txt","text2.txt"]
  return files
def read_file(args):
  return "respond: "+args


In [5]:
import json
iteration =0
max_iteration=2

print("what do you want to know abut files? if you want about directory give me a directory name(academic /code)")

#the agent loop
while iteration<max_iteration:
  message=input("action: \n ")
  memory.extend([
    {"role":"user","content":message},
   ])
  prompt=agent_rules+memory
  print("\n agent thinking.......\n ")

  response=generate_response(prompt)
  print(f"Agent response: {response}")
  action=parse_action(response)

  result="Action Executed"
  if action["tool_name"]=="list_files":
    result={"result":list_file()}
  elif action["tool_name"]=="read_file":
    result={"result": read_file(action["args"]["file_name"])}
  elif action["tool_name"] == "error":
    result = {"error": action["args"]["message"]}
  elif action["tool_name"] == "terminate":
    print(action["args"]["message"])
    break
  else :
    result={"error ": "Unknown action"+action["tool_name"]}
  print(f"Action result : {result}")

  #update memory
  memory.extend([
      {"role":"assistant", "content":response},
      {"role":"user","content":json.dumps(result)}
  ])

  iteration+=1

what do you want to know abut files? if you want about directory give me a directory name(academic /code)
action: 
 academic

 agent thinking.......
 
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result : {'result': ['text1.txt', 'text2.txt']}
action: 
 text1.txt

 agent thinking.......
 
Agent response: ```action
{
    "tool_name": "read_file",
    "args": {
        "file_name": "text1.txt"
    }
}
```
Action result : {'result': 'respond: text1.txt'}
